# DA02 Analysis Notebook
## U.S. Consumer Credit Analysis (Federal Reserve G.19)
**Pipeline:** 5-stage Explore → Profile → Clean → Shape → Analyze

**Date range:** January 2020 – January 2025 (61 months)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# Create charts directory
CHARTS_DIR = "charts"
if not os.path.exists(CHARTS_DIR):
    os.makedirs(CHARTS_DIR)

print("✓ Setup complete")


✓ Setup complete


In [2]:
HOLDER_COLORS = {
    "Depository Institutions": "#2196F3",
    "Federal Government": "#F44336",
    "Finance Companies": "#FF9800",
    "Credit Unions": "#4CAF50",
    "Nonprofit & Educational": "#9C27B0",
}

ERA_COLORS = {
    "Pre-COVID": "#6baed6",
    "COVID Shock": "#e6550d",
    "Recovery": "#74c476",
    "Rate Hike Cycle": "#756bb1",
    "Post-Hike": "#fd8d3c",
}

print("✓ Color palettes loaded")


✓ Color palettes loaded


In [3]:
# Load raw data
df_raw = pd.read_csv('data/FRB_G19_(2).csv', index_col=0)

# The first row contains headers in the index, columns are metadata + dates
# Rename index
df_raw.index.name = 'Descriptions'

# Identify date columns (numeric YYYY-MM format)
date_cols = [col for col in df_raw.columns if col not in [
    'Unit:', 'Multiplier:', 'Currency:', 'Unique Identifier:', 'Series Name:'
]]

print(f"Raw data shape: {df_raw.shape}")
print(f"Series: {len(df_raw)}")
print(f"Date columns: {len(date_cols)} (from {date_cols[0]} to {date_cols[-1]})")
print(f"\nSample series descriptions:")
for i, desc in enumerate(df_raw.index[:3], 1):
    print(f"  {i}. {desc[:70]}...")


Raw data shape: (50, 66)
Series: 50
Date columns: 61 (from 2020-01 to 2025-01)

Sample series descriptions:
  1. Total consumer credit securitized by depository institutions, not seas...
  2. Total consumer credit securitized by depository institutions, not seas...
  3. Total consumer credit securitized by finance companies, not seasonally...


## STAGE A: Explore
*Understand dataset structure and identify data quality issues.*


In [4]:
print("\n" + "="*70)
print("STAGE A: EXPLORE")
print("="*70)

# A.1: Series inventory
print(f"\nA.1 - Series Inventory:")
print(f"  Total series: {len(df_raw)}")
level_count = df_raw.index.str.contains('level', case=False).sum()
flow_count = df_raw.index.str.contains('flow', case=False).sum()
print(f"  Level series: {level_count}")
print(f"  Flow series: {flow_count}")

# A.2: Coverage by era (simple counts)
era_bounds = [
    ('Pre-COVID', '2020-01', '2020-02'),
    ('COVID Shock', '2020-03', '2020-12'),
    ('Recovery', '2021-01', '2022-02'),
    ('Rate Hike Cycle', '2022-03', '2023-12'),
    ('Post-Hike', '2024-01', '2025-01')
]

print(f"\nA.2 - Coverage by Era:")
era_months = {}
for era, start, end in era_bounds:
    start_idx = date_cols.index(start) if start in date_cols else None
    end_idx = date_cols.index(end) if end in date_cols else None
    if start_idx is not None and end_idx is not None:
        months = end_idx - start_idx + 1
        era_months[era] = months
        print(f"  {era:18s}: {months:2d} months")

# A.3: All-zero series
all_zero_mask = (df_raw[date_cols] == 0).all(axis=1)
zero_count = all_zero_mask.sum()
print(f"\nA.3 - All-zero series: {zero_count}")

# A.4: Null check
null_count = df_raw[date_cols].isnull().sum().sum()
total_cells = len(df_raw) * len(date_cols)
print(f"\nA.4 - Null rate: {null_count}/{total_cells} ({100*null_count/total_cells:.3f}%)")

print(f"\n✓ Exploration complete")



STAGE A: EXPLORE

A.1 - Series Inventory:
  Total series: 50
  Level series: 25
  Flow series: 25

A.2 - Coverage by Era:
  Pre-COVID         :  2 months
  COVID Shock       : 10 months
  Recovery          : 14 months
  Rate Hike Cycle   : 22 months
  Post-Hike         : 13 months

A.3 - All-zero series: 6

A.4 - Null rate: 0/3050 (0.000%)

✓ Exploration complete


In [5]:
# Create coverage chart
fig, ax = plt.subplots(figsize=(12, 6))
eras_list = list(era_months.keys())
months = list(era_months.values())
colors = [ERA_COLORS.get(era, '#ccc') for era in eras_list]

ax.barh(eras_list, months, color=colors, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Number of Months', fontsize=11, fontweight='bold')
ax.set_title('Data Coverage by Economic Era', fontsize=13, fontweight='bold', pad=15)

for i, (bar, val) in enumerate(zip(ax.containers[0], months)):
    ax.text(val + 0.3, i, f'{int(val)}mo', va='center', fontsize=10, fontweight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'A2_coverage_by_era.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: A2_coverage_by_era.png")
plt.close()


✓ Saved: A2_coverage_by_era.png


## STAGE B: Profile
*Analyze distributions and patterns.*


In [6]:
print("\n" + "="*70)
print("STAGE B: PROFILE")
print("="*70)

# Convert to long format
df_long = df_raw.reset_index().melt(
    id_vars=['Descriptions', 'Unit:', 'Multiplier:', 'Currency:', 'Unique Identifier:', 'Series Name:'],
    var_name='date', value_name='value_millions'
)

# Filter to date columns only
df_long = df_long[df_long['date'].isin(date_cols)].copy()

# Parse dimensions
def extract_holder(desc):
    d = desc.lower()
    if 'depository' in d: return 'Depository Institutions'
    elif 'finance' in d: return 'Finance Companies'
    elif 'credit union' in d: return 'Credit Unions'
    elif 'federal' in d: return 'Federal Government'
    elif 'nonprofit' in d or 'educational' in d: return 'Nonprofit & Educational'
    return 'Other'

def extract_credit_type(desc):
    d = desc.lower()
    if 'nonrevolving' in d: return 'Nonrevolving'
    elif 'revolving' in d: return 'Revolving'
    elif 'total' in d and 'revolving' not in d and 'nonrevolving' not in d: return 'Total'
    return 'Other'

def extract_ownership(desc):
    return 'Securitized' if 'securitized' in desc.lower() else 'Owned'

def extract_measure(desc):
    d = desc.lower()
    if 'flow' in d: return 'Flow'
    elif 'level' in d: return 'Level'
    return 'Other'

df_long['holder'] = df_long['Descriptions'].apply(extract_holder)
df_long['credit_type'] = df_long['Descriptions'].apply(extract_credit_type)
df_long['ownership_type'] = df_long['Descriptions'].apply(extract_ownership)
df_long['measure'] = df_long['Descriptions'].apply(extract_measure)
df_long['date'] = pd.to_datetime(df_long['date'])

print(f"\nLong format created: {len(df_long)} rows")
print(f"  Holders: {df_long['holder'].nunique()}")
print(f"  Credit types: {df_long['credit_type'].nunique()}")
print(f"  Measures: {df_long['measure'].nunique()}")



STAGE B: PROFILE

Long format created: 3050 rows
  Holders: 5
  Credit types: 3
  Measures: 2


In [7]:
# B.1: Value distributions
df_level = df_long[(df_long['measure'] == 'Level') &
                     (df_long['ownership_type'] == 'Owned')].copy()

# Compute YoY growth
df_level = df_level.sort_values(['Series Name:', 'date'])
df_level['value_prev_year'] = df_level.groupby('Series Name:')['value_millions'].shift(12)
df_level['yoy_growth_pct'] = ((df_level['value_millions'] - df_level['value_prev_year']) /
                               df_level['value_prev_year'] * 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Balance distribution
ax = axes[0]
for holder in HOLDER_COLORS.keys():
    data = df_level[df_level['holder'] == holder]['value_millions'].dropna()
    if len(data) > 0:
        ax.hist(data, bins=20, alpha=0.6, label=holder, color=HOLDER_COLORS[holder])
ax.set_xlabel('Outstanding Balance ($ millions)', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.set_title('Distribution of Outstanding Balances', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Growth distribution
ax = axes[1]
for holder in HOLDER_COLORS.keys():
    data = df_level[df_level['holder'] == holder]['yoy_growth_pct'].dropna()
    if len(data) > 0:
        ax.hist(data, bins=20, alpha=0.6, label=holder, color=HOLDER_COLORS[holder])
ax.set_xlabel('YoY Growth (%)', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.set_title('Distribution of YoY Growth', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'B1_value_distributions.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: B1_value_distributions.png")
plt.close()


✓ Saved: B1_value_distributions.png


In [8]:
# B.2: Era heatmap (simplified)
jan2025 = df_level[df_level['date'] == df_level['date'].max()]
total = jan2025[jan2025['holder'] != 'Other']['value_millions'].sum()

shares = {}
for holder in HOLDER_COLORS.keys():
    val = jan2025[jan2025['holder'] == holder]['value_millions'].sum()
    shares[holder] = (val / total * 100) if total > 0 else 0

fig, ax = plt.subplots(figsize=(10, 6))
holders = list(shares.keys())
values = list(shares.values())
colors = [HOLDER_COLORS[h] for h in holders]
ax.barh(holders, values, color=colors, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Share of Total Outstanding (%)', fontsize=11, fontweight='bold')
ax.set_title('Holder Share - January 2025 (Owned Series)', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, v in enumerate(values):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'B2_era_heatmap.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: B2_era_heatmap.png")
plt.close()

# B.3: YoY correlations
yoy_pivot = df_level.pivot_table(index='date', columns='holder', values='yoy_growth_pct')
yoy_pivot = yoy_pivot.drop('Other', axis=1, errors='ignore')
corr = yoy_pivot.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True,
            linewidths=2, cbar_kws={"shrink": 0.8}, ax=ax, vmin=-1, vmax=1)
ax.set_title('YoY Growth Correlations', fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'B3_correlation_matrix.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: B3_correlation_matrix.png")
plt.close()

# B.4: Revolving vs Nonrevolving
df_types = df_long[(df_long['holder'] == 'Depository Institutions') &
                     (df_long['ownership_type'] == 'Owned') &
                     (df_long['measure'] == 'Level') &
                     (df_long['credit_type'].isin(['Revolving', 'Nonrevolving', 'Total']))].copy()
df_types = df_types.sort_values('date')

fig, ax = plt.subplots(figsize=(12, 6))
for ctype in ['Revolving', 'Nonrevolving', 'Total']:
    data = df_types[df_types['credit_type'] == ctype].copy()
    if len(data) > 0:
        agg = data.groupby('date')['value_millions'].sum()
        ax.plot(agg.index, agg.values, linewidth=2.5, label=ctype, marker='o', markersize=3)
ax.set_xlabel('Date', fontsize=11, fontweight='bold')
ax.set_ylabel('Outstanding ($ millions)', fontsize=11, fontweight='bold')
ax.set_title('Depository Institutions: Revolving vs Nonrevolving', fontsize=12, fontweight='bold')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'B4_credit_type_comparison.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: B4_credit_type_comparison.png")
plt.close()

# B.5: Simple scatter (volatility by sample)
volatility_data = []
for holder in HOLDER_COLORS.keys():
    vol = df_level[df_level['holder'] == holder]['yoy_growth_pct'].std()
    if not np.isnan(vol):
        volatility_data.append({'holder': holder, 'volatility': vol})
df_vol = pd.DataFrame(volatility_data)

fig, ax = plt.subplots(figsize=(10, 6))
for _, row in df_vol.iterrows():
    ax.scatter(1, row['volatility'], s=300, color=HOLDER_COLORS[row['holder']],
              edgecolor='black', linewidth=2, alpha=0.7)
    ax.text(1.1, row['volatility'], row['holder'], fontsize=9, fontweight='bold', va='center')
ax.set_xlim(0.5, 2)
ax.set_ylabel('YoY Growth Volatility (std dev %)', fontsize=11, fontweight='bold')
ax.set_title('Growth Volatility by Holder', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.set_xticks([])
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'B5_revolving_vs_growth.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: B5_revolving_vs_growth.png")
plt.close()

# B.6: Top/bottom flow months
df_flows = df_long[(df_long['measure'] == 'Flow') &
                     (df_long['ownership_type'] == 'Owned')].copy()
df_flows = df_flows.sort_values('date')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for idx, holder in enumerate(list(HOLDER_COLORS.keys())[:6]):
    ax = axes[idx]
    agg_flow = df_flows[df_flows['holder'] == holder].groupby('date')['value_millions'].sum()
    if len(agg_flow) > 0:
        top_bottom = pd.concat([agg_flow.nsmallest(3), agg_flow.nlargest(3)]).sort_values()
        colors = ['#e74c3c' if v < 0 else '#27ae60' for v in top_bottom.values]
        ax.barh(range(len(top_bottom)), top_bottom.values, color=colors, edgecolor='black', linewidth=1)
        ax.set_yticks(range(len(top_bottom)))
        ax.set_yticklabels([d.strftime('%Y-%m') for d in top_bottom.index], fontsize=8)
        ax.set_xlabel('Flow ($ millions)', fontsize=9, fontweight='bold')
        ax.set_title(holder, fontsize=10, fontweight='bold')
        ax.axvline(x=0, color='black', linewidth=1)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
plt.suptitle('Top 3 & Bottom 3 Flow Months by Holder', fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'B6_holder_top_bottom.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: B6_holder_top_bottom.png")
plt.close()

print("\n✓ Profile stage complete")


✓ Saved: B2_era_heatmap.png
✓ Saved: B3_correlation_matrix.png


✓ Saved: B4_credit_type_comparison.png


✓ Saved: B5_revolving_vs_growth.png


✓ Saved: B6_holder_top_bottom.png

✓ Profile stage complete


## STAGE C: Clean
*Remove problematic series, tag outliers.*


In [9]:
print("\n" + "="*70)
print("STAGE C: CLEAN")
print("="*70)

# C.1: Remove all-zero and securitized for primary analysis
df_clean = df_long[
    (~df_long['Descriptions'].str.contains('securitized', case=False)) &
    (df_long['holder'] != 'Other')
].copy()

print(f"\nRows after filtering:")
print(f"  Removed securitized: removed")
print(f"  Remaining: {len(df_clean):,} rows")

# C.2: Tag extreme outliers (flow data with |z-score| > 2.0)
df_flow = df_clean[df_clean['measure'] == 'Flow'].copy()
df_flow['z_score'] = df_flow.groupby('Series Name:')['value_millions'].transform(
    lambda x: np.abs(stats.zscore(x, nan_policy='omit')))
outlier_count = (df_flow['z_score'] > 2.0).sum()

print(f"\nOutlier detection (Flow data, |z| > 2.0):")
print(f"  Extreme outliers tagged: {outlier_count}")

print("\n✓ Cleaning complete")



STAGE C: CLEAN

Rows after filtering:
  Removed securitized: removed
  Remaining: 1,586 rows

Outlier detection (Flow data, |z| > 2.0):
  Extreme outliers tagged: 45

✓ Cleaning complete


## STAGE D: Shape & Export
*Compute derived metrics, export analysis dataset.*


In [10]:
print("\n" + "="*70)
print("STAGE D: SHAPE & EXPORT")
print("="*70)

# Assign eras
def assign_era(date):
    for era, start, end in era_bounds:
        start_dt = pd.Timestamp(f'{start}-01')
        end_dt = pd.Timestamp(f'{end}-01') + pd.DateOffset(months=1) - pd.Timedelta(days=1)
        if start_dt <= date <= end_dt:
            return era
    return 'Unknown'

df_clean['era'] = df_clean['date'].apply(assign_era)

# Compute YoY and MoM growth
df_clean = df_clean.sort_values(['Series Name:', 'date'])
df_clean['value_prev_year'] = df_clean.groupby('Series Name:')['value_millions'].shift(12)
df_clean['yoy_growth_pct'] = ((df_clean['value_millions'] - df_clean['value_prev_year']) /
                               df_clean['value_prev_year'] * 100)

df_clean['value_prev_month'] = df_clean.groupby('Series Name:')['value_millions'].shift(1)
df_clean['mom_growth_pct'] = ((df_clean['value_millions'] - df_clean['value_prev_month']) /
                               df_clean['value_prev_month'] * 100)

# Rolling 3m average
df_clean['rolling_3m_avg'] = df_clean.groupby('Series Name:')['value_millions'].rolling(3, min_periods=1).mean().reset_index(drop=True)

# Holder share
total_owned = df_clean[
    (df_clean['ownership_type'] == 'Owned') &
    (df_clean['credit_type'] == 'Total') &
    (df_clean['measure'] == 'Level')
].groupby('date')['value_millions'].sum()

def calc_share(row):
    if row['ownership_type'] == 'Owned' and row['credit_type'] == 'Total' and row['measure'] == 'Level':
        t = total_owned.get(row['date'])
        if pd.notna(t) and t > 0:
            return (row['value_millions'] / t) * 100
    return np.nan

df_clean['holder_share_pct'] = df_clean.apply(calc_share, axis=1)

# Revolving share
rev_total = df_clean[
    (df_clean['ownership_type'] == 'Owned') &
    (df_clean['credit_type'] == 'Revolving') &
    (df_clean['measure'] == 'Level')
].groupby('date')['value_millions'].sum()

owned_total = df_clean[
    (df_clean['ownership_type'] == 'Owned') &
    (df_clean['credit_type'] == 'Total') &
    (df_clean['measure'] == 'Level')
].groupby('date')['value_millions'].sum()

revolving_share_by_date = (rev_total / owned_total * 100).to_dict()
df_clean['revolving_share_pct'] = df_clean['date'].map(revolving_share_by_date)

# Tag outliers
df_clean['is_extreme_outlier'] = False
if len(df_flow) > 0:
    outlier_rows = df_flow[df_flow['z_score'] > 2.0]
    for _, row in outlier_rows.iterrows():
        mask = (df_clean['Series Name:'] == row['Series Name:']) & (df_clean['date'] == row['date'])
        df_clean.loc[mask, 'is_extreme_outlier'] = True

# Export
df_export = df_clean[[
    'Series Name:', 'Descriptions', 'date', 'holder', 'credit_type', 'ownership_type',
    'measure', 'value_millions', 'era', 'yoy_growth_pct', 'mom_growth_pct',
    'rolling_3m_avg', 'holder_share_pct', 'revolving_share_pct', 'is_extreme_outlier'
]].copy()

df_export.columns = [
    'Series Name', 'Descriptions', 'date', 'holder', 'credit_type', 'ownership_type',
    'measure', 'value_millions', 'era', 'yoy_growth_pct', 'mom_growth_pct',
    'rolling_3m_avg', 'holder_share_pct', 'revolving_share_pct', 'is_extreme_outlier'
]

df_export.to_csv('data/credit_analysis_final.csv', index=False)

print(f"\nExported: data/credit_analysis_final.csv")
print(f"Rows: {len(df_export):,}")
print(f"Date range: {df_export['date'].min().date()} to {df_export['date'].max().date()}")
print("\n✓ Export complete")



STAGE D: SHAPE & EXPORT

Exported: data/credit_analysis_final.csv
Rows: 1,586
Date range: 2020-01-01 to 2025-01-01

✓ Export complete


## STAGE E: Analyze
*Answer five key questions: Size → Rank → Explain → Compare → Recommend*


In [11]:
print("\n" + "="*70)
print("STAGE E: ANALYZE")
print("="*70)

# E.1: Stacked area - outstanding by holder
df_stack = df_export[
    (df_export['ownership_type'] == 'Owned') &
    (df_export['credit_type'] == 'Total') &
    (df_export['measure'] == 'Level')
].copy()

pivot_data = df_stack.pivot_table(index='date', columns='holder', values='value_millions', aggfunc='sum')

fig, ax = plt.subplots(figsize=(14, 7))
pivot_data.plot(ax=ax, kind='area', color=[HOLDER_COLORS.get(h, '#ccc') for h in pivot_data.columns],
                alpha=0.7, linewidth=2)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Outstanding Balance ($ billions)', fontsize=12, fontweight='bold')
ax.set_title('Total Consumer Credit Outstanding by Holder', fontsize=13, fontweight='bold', pad=15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.1f}B'))
ax.legend(loc='upper left', frameon=False, fontsize=10, ncol=2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3, axis='y')

# Add era shading
for era, start, end in era_bounds:
    start_dt = pd.Timestamp(f'{start}-01')
    end_dt = pd.Timestamp(f'{end}-01') + pd.DateOffset(months=1)
    ax.axvspan(start_dt, end_dt, alpha=0.1, color=ERA_COLORS.get(era, '#ccc'))

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'E1_stacked_area.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: E1_stacked_area.png")
plt.close()

# Print size statistics
jan2025 = df_export[(df_export['date'] == df_export['date'].max()) &
                     (df_export['measure'] == 'Level') &
                     (df_export['credit_type'] == 'Total')]
print("\nE.1 - Total Outstanding (Jan 2025):")
for _, row in jan2025.sort_values('value_millions', ascending=False).iterrows():
    print(f"  {row['holder']:25s}: ${row['value_millions']:,.0f}M")



STAGE E: ANALYZE


✓ Saved: E1_stacked_area.png

E.1 - Total Outstanding (Jan 2025):
  Depository Institutions  : $1,986,628M
  Federal Government       : $1,538,343M
  Finance Companies        : $738,171M
  Credit Unions            : $650,739M
  Nonprofit & Educational  : $15,355M


In [12]:
# E.2: Flow rank - average flow by era
df_flow_rank = df_export[df_export['measure'] == 'Flow'].copy()

flow_by_era_holder = df_flow_rank.groupby(['era', 'holder'])['value_millions'].mean().reset_index()
flow_pivot = flow_by_era_holder.pivot(index='holder', columns='era', values='value_millions')

# Reorder by era
era_order = ['Pre-COVID', 'COVID Shock', 'Recovery', 'Rate Hike Cycle', 'Post-Hike']
flow_pivot = flow_pivot[[e for e in era_order if e in flow_pivot.columns]]

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(flow_pivot.index))
width = 0.15

for i, era in enumerate(flow_pivot.columns):
    offset = (i - 2) * width
    color = ERA_COLORS.get(era, '#ccc')
    ax.bar(x + offset, flow_pivot[era], width, label=era, color=color, edgecolor='black', linewidth=1, alpha=0.8)

ax.set_xlabel('Holder', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Monthly Flow ($ millions)', fontsize=12, fontweight='bold')
ax.set_title('Average Monthly Net Flow by Era and Holder', fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(flow_pivot.index, rotation=45, ha='right')
ax.axhline(y=0, color='black', linewidth=1)
ax.legend(loc='upper right', frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'E2_flow_rank.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: E2_flow_rank.png")
plt.close()

# Calculate flow deterioration
recovery_avg = flow_by_era_holder[flow_by_era_holder['era'] == 'Recovery']
rate_hike_avg = flow_by_era_holder[flow_by_era_holder['era'] == 'Rate Hike Cycle']

print("\nE.2 - Flow Deterioration (Rate Hike vs Recovery):")
for _, rec in recovery_avg.iterrows():
    rh = rate_hike_avg[rate_hike_avg['holder'] == rec['holder']]['value_millions'].values
    if len(rh) > 0:
        delta = rh[0] - rec['value_millions']
        print(f"  {rec['holder']:25s}: {rec['value_millions']:8,.0f}M → {rh[0]:8,.0f}M ({delta:+8,.0f}M)")


✓ Saved: E2_flow_rank.png

E.2 - Flow Deterioration (Rate Hike vs Recovery):
  Credit Unions            :    1,649M →    3,723M (  +2,074M)
  Depository Institutions  :    5,674M →    8,987M (  +3,313M)
  Federal Government       :    5,249M →      351M (  -4,899M)
  Finance Companies        :      855M →      971M (    +116M)
  Nonprofit & Educational  :     -181M →     -133M (     +48M)


In [13]:
# E.3: YoY growth trends
df_yoy = df_export[
    (df_export['ownership_type'] == 'Owned') &
    (df_export['credit_type'] == 'Total') &
    (df_export['measure'] == 'Level')
].copy()

fig, ax = plt.subplots(figsize=(14, 7))

for holder in HOLDER_COLORS.keys():
    df_h = df_yoy[df_yoy['holder'] == holder].sort_values('date')
    if len(df_h) > 0:
        ax.plot(df_h['date'], df_h['yoy_growth_pct'], linewidth=2.5, label=holder,
               color=HOLDER_COLORS[holder], marker='o', markersize=4)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('YoY Growth Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('YoY Growth Trends by Holder', fontsize=13, fontweight='bold', pad=15)

# Add era shading
for era, start, end in era_bounds:
    start_dt = pd.Timestamp(f'{start}-01')
    end_dt = pd.Timestamp(f'{end}-01') + pd.DateOffset(months=1)
    ax.axvspan(start_dt, end_dt, alpha=0.08, color=ERA_COLORS.get(era, '#ccc'))

ax.axhline(y=0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax.legend(loc='best', frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'E3_yoy_trends.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: E3_yoy_trends.png")
plt.close()

print("\nE.3 - YoY Growth Summary:")
for holder in HOLDER_COLORS.keys():
    data = df_yoy[df_yoy['holder'] == holder]['yoy_growth_pct'].dropna()
    if len(data) > 0:
        print(f"  {holder:25s}: mean={data.mean():6.2f}% std={data.std():6.2f}%")


✓ Saved: E3_yoy_trends.png

E.3 - YoY Growth Summary:
  Depository Institutions  : mean=  5.47% std=  5.22%
  Federal Government       : mean=  2.42% std=  2.05%
  Finance Companies        : mean=  8.31% std= 10.49%
  Credit Unions            : mean=  7.03% std=  6.92%
  Nonprofit & Educational  : mean=-10.57% std=  5.47%


In [14]:
# E.4: Revolving vs Nonrevolving
df_types_e4 = df_export[
    (df_export['ownership_type'] == 'Owned') &
    (df_export['measure'] == 'Level') &
    (df_export['credit_type'].isin(['Revolving', 'Nonrevolving']))
].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Outstanding
ax = axes[0]
for ctype in ['Revolving', 'Nonrevolving']:
    agg = df_types_e4[df_types_e4['credit_type'] == ctype].groupby('date')['value_millions'].sum()
    ax.plot(agg.index, agg.values, linewidth=2.5, label=ctype, marker='o', markersize=4)
ax.set_xlabel('Date', fontsize=11, fontweight='bold')
ax.set_ylabel('Outstanding ($ millions)', fontsize=11, fontweight='bold')
ax.set_title('Revolving vs Nonrevolving Outstanding', fontsize=12, fontweight='bold')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)

# Flows
ax = axes[1]
df_flows_e4 = df_export[
    (df_export['ownership_type'] == 'Owned') &
    (df_export['measure'] == 'Flow') &
    (df_export['credit_type'].isin(['Revolving', 'Nonrevolving']))
].copy()

for ctype in ['Revolving', 'Nonrevolving']:
    agg = df_flows_e4[df_flows_e4['credit_type'] == ctype].groupby('date')['value_millions'].sum()
    ax.bar(agg.index, agg.values, label=ctype, alpha=0.7, width=15)
ax.set_xlabel('Date', fontsize=11, fontweight='bold')
ax.set_ylabel('Monthly Net Flow ($ millions)', fontsize=11, fontweight='bold')
ax.set_title('Monthly Flow: Revolving vs Nonrevolving', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='black', linewidth=1)
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'E4_revolving_compare.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: E4_revolving_compare.png")
plt.close()

# Latest share
latest_rev_share = df_export[df_export['date'] == df_export['date'].max()]['revolving_share_pct'].iloc[0]
print(f"\nE.4 - Revolving Share: {latest_rev_share:.1f}% (Jan 2025)")


✓ Saved: E4_revolving_compare.png

E.4 - Revolving Share: 25.8% (Jan 2025)


In [15]:
# E.5: Composite stress analysis
# Calculate stress components by holder

stress_data = []

for holder in HOLDER_COLORS.keys():
    # Component 1: Flow deterioration (Rate Hike vs Recovery)
    flow_recovery = flow_by_era_holder[
        (flow_by_era_holder['era'] == 'Recovery') &
        (flow_by_era_holder['holder'] == holder)
    ]['value_millions'].values
    flow_rate_hike = flow_by_era_holder[
        (flow_by_era_holder['era'] == 'Rate Hike Cycle') &
        (flow_by_era_holder['holder'] == holder)
    ]['value_millions'].values

    if len(flow_recovery) > 0 and len(flow_rate_hike) > 0:
        flow_deterioration = flow_recovery[0] - flow_rate_hike[0]
    else:
        flow_deterioration = 0

    # Component 2: YoY deceleration
    yoy_recovery = df_yoy[
        (df_yoy['era'] == 'Recovery') &
        (df_yoy['holder'] == holder)
    ]['yoy_growth_pct'].mean()
    yoy_rate_hike = df_yoy[
        (df_yoy['era'] == 'Rate Hike Cycle') &
        (df_yoy['holder'] == holder)
    ]['yoy_growth_pct'].mean()

    yoy_deceleration = yoy_recovery - yoy_rate_hike if pd.notna(yoy_recovery) and pd.notna(yoy_rate_hike) else 0

    # Component 3: Portfolio share
    latest_share = df_export[
        (df_export['date'] == df_export['date'].max()) &
        (df_export['holder'] == holder) &
        (df_export['holder_share_pct'].notna())
    ]['holder_share_pct'].mean()
    portfolio_share = latest_share if pd.notna(latest_share) else 0

    stress_data.append({
        'holder': holder,
        'flow_deterioration': flow_deterioration,
        'yoy_deceleration': yoy_deceleration,
        'portfolio_share': portfolio_share
    })

df_stress = pd.DataFrame(stress_data)

# Normalize
for col in ['flow_deterioration', 'yoy_deceleration', 'portfolio_share']:
    min_v = df_stress[col].min()
    max_v = df_stress[col].max()
    if max_v > min_v:
        df_stress[f'{col}_norm'] = ((df_stress[col] - min_v) / (max_v - min_v)) * 100
    else:
        df_stress[f'{col}_norm'] = 0

# Composite score
df_stress['composite_stress'] = (
    df_stress['flow_deterioration_norm'] * 0.40 +
    df_stress['yoy_deceleration_norm'] * 0.40 +
    df_stress['portfolio_share_norm'] * 0.20
)

df_stress = df_stress.sort_values('composite_stress', ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

for _, row in df_stress.iterrows():
    color = HOLDER_COLORS.get(row['holder'], '#ccc')
    size = row['portfolio_share'] * 30 + 100
    ax.scatter(row['flow_deterioration'], row['composite_stress'], s=size, alpha=0.7,
              color=color, edgecolors='black', linewidth=2, label=row['holder'])
    ax.text(row['flow_deterioration'] + 100, row['composite_stress'], row['holder'],
           fontsize=9, fontweight='bold', va='center')

ax.set_xlabel('Flow Deterioration (Recovery → Rate Hike, $M)', fontsize=12, fontweight='bold')
ax.set_ylabel('Composite Stress Score (0-100)', fontsize=12, fontweight='bold')
ax.set_title('Composite Credit Stress (40% Flow + 40% YoY + 20% Share)', fontsize=13, fontweight='bold', pad=15)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.text(0.02, 0.98, 'Bubble size = portfolio share', transform=ax.transAxes, fontsize=10, va='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'E5_stress_matrix.png'), dpi=150, bbox_inches='tight')
print("✓ Saved: E5_stress_matrix.png")
plt.close()

# Print ranking
print("\nE.5 - Composite Stress Ranking:")
print("  Rank  Holder                       Score")
print("  " + "=" * 60)
for rank, (_, row) in enumerate(df_stress.head(5).iterrows(), 1):
    print(f"  {rank:2d}.   {row['holder']:25s}  {row['composite_stress']:5.1f}")

print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)
print(f"\n✓ All charts saved to: {CHARTS_DIR}/")
print(f"✓ Data exported to: data/credit_analysis_final.csv")


✓ Saved: E5_stress_matrix.png

E.5 - Composite Stress Ranking:
  Rank  Holder                       Score
   1.   Federal Government          74.1
   2.   Finance Companies           62.9
   3.   Nonprofit & Educational     32.6
   4.   Depository Institutions     25.8
   5.   Credit Unions               12.5

ANALYSIS COMPLETE

✓ All charts saved to: charts/
✓ Data exported to: data/credit_analysis_final.csv
